In [11]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [12]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

Q1. How many lesson pages

In [3]:
len(documents)

72

Q2. Indexing and searching

In [4]:
from minsearch import Index

index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(documents)

In [5]:
question = "How does the agentic loop keep calling the model until it stops?"

search_results = index.search(
    question,
    num_results=1
)

search_results

[{'content': '# The Agentic Loop\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we did function calling by hand. We sent a\nmessage and got back a function call. We ran it, sent the result back,\nand got the answer.\n\nThat works for one function call. It breaks down when the model wants\nto search several times, or when the first search misses the answer.\nWe don\'t know in advance how many calls the model will want. So we\nneed a loop that keeps calling the model and running tools until it\'s\ndone. An agent is exactly that.\n\n## Anatomy of an agent\n\nWith the LLM in the driver\'s seat, we have an agent. It\'s an AI\nassistant whose goal is to help the user.\n\nAn agent has three parts:\n\n- Instructions, the role and behavior we want. We pass this as the\n  `developer` message. The better the instructions, the better the\n  agent helps.\n- Tools, the functions the agent can call to carry

Q3. RAG

In [6]:
from dotenv import load_dotenv
load_dotenv()

import os
from rag_helper import RAGBase
from google import genai

client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

assistant = RAGBase(
    index=index,
    llm_client=client
)

In [7]:
answer = assistant.rag("How does the agentic loop keep calling the model until it stops?")
print(answer)

print(f"Input tokens: {assistant.last_response.usage_metadata.prompt_token_count}")

The agentic loop keeps calling the model by using a `while` loop that continues to execute as long as the model returns a response containing a function call.

Here is how the mechanism works:

1.  **The Loop:** A `while True` loop is used to repeatedly send the conversation history (including previous tool outputs) to the model.
2.  **Function Call Execution:** Inside the loop, the code checks the model's response for `function_call` items. If found, it uses a helper function to execute the requested tool and appends the tool's output to the `messages` list.
3.  **Tracking State:** A flag (such as `has_function_calls`) is set to `True` whenever a function call is detected in a response.
4.  **Termination Condition:** After processing the response, the loop checks the flag. If no function calls were made by the model in the current turn, the `has_function_calls` flag remains `False`, and the loop `break`s, signaling that the model is finished and has provided a final answer. 

In short

Q4. Chunking

In [13]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [11]:
len(chunks)

295

Q5. RAG with chunking

In [14]:
from minsearch import Index

index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(chunks)

from dotenv import load_dotenv
load_dotenv()

import os
from rag_helper import RAGBase
from google import genai

client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

assistant = RAGBase(
    index=index,
    llm_client=client
)

answer = assistant.rag("How does the agentic loop keep calling the model until it stops?")
print(answer)

print(f"Input tokens: {assistant.last_response.usage_metadata.prompt_token_count}")

The agentic loop uses a `while True` loop to keep calling the model. Inside this loop, the model generates a response based on the message history (which includes instructions, previous tool results, and the original question).

The loop continues based on the following logic:

*   **Processing responses:** The agent checks the model's output for `function_call` types. If a function call is found, the code executes it, appends the result to the message history, and sets a flag (`has_function_calls`) to `True`.
*   **The exit condition:** After processing the model's output for the current turn, the loop checks the `has_function_calls` flag. If the model does not request any function calls (i.e., the flag remains `False`), it means the model has provided a final answer, and the loop executes `break` to exit. 

In short, the model decides how many times it needs to search, and the loop keeps running until the model stops asking for tools.
Input tokens: 2600


Q6. Turning it into an agent

In [ ]:
def search(query: str) -> list:
    """Search the chunk index database for entries matching the given query.

    Args:
        query: The search query text to look up in the document chunks.

    {
        "cells": [
            {
                "cell_type": "code",
                "metadata": { "id": "#VSC-3d953b13", "language": "python" },
                "source": [
                    "from gitsource import GithubRepositoryDataReader\n",
                    "\n",
                    "reader = GithubRepositoryDataReader(\n",
                    "    repo_owner=\"DataTalksClub\",\n",
                    "    repo_name=\"llm-zoomcamp\",\n",
                    "    commit_id=\"8c1834d\",\n",
                    "    allowed_extensions={\"md\"},\n",
                    "    filename_filter=lambda path: \"/lessons/\" in path,\n",
                    ")\n",
                    "\n",
                    "files = reader.read()\n"
                ],
                "execution_count": null,
                "outputs": []
            },
            {
                "cell_type": "code",
                "metadata": { "id": "#VSC-19d51e0e", "language": "python" },
                "source": [
                    "documents = []\n",
                    "\n",
                    "for file in files:\n",
                    "    doc = file.parse()\n",
                    "    documents.append(doc)\n"
                ],
                "execution_count": null,
                "outputs": []
            },
            {
                "cell_type": "markdown",
                "metadata": { "id": "#VSC-4e877645", "language": "markdown" },
                "source": ["Q1. How many lesson pages\n"]
            },
            {
                "cell_type": "code",
                "metadata": { "id": "#VSC-a1521333", "language": "python" },
                "source": ["len(documents)\n"],
                "execution_count": null,
                "outputs": []
            },
            {
                "cell_type": "markdown",
                "metadata": { "id": "#VSC-b1943f72", "language": "markdown" },
                "source": ["Q2. Indexing and searching\n"]
            },
            {
                "cell_type": "code",
                "metadata": { "id": "#VSC-028a72d2", "language": "python" },
                "source": [
                    "from minsearch import Index\n",
                    "\n",
                    "index = Index(\n",
                    "    text_fields=[\"content\"],\n",
                    "    keyword_fields=[\"filename\"]\n",
                    ")\n",
                    "\n",
                    "index.fit(documents)\n"
                ],
                "execution_count": null,
                "outputs": []
            },
            {
                "cell_type": "code",
                "metadata": { "id": "#VSC-728ddcbd", "language": "python" },
                "source": [
                    "question = \"How does the agentic loop keep calling the model until it stops?\"\n",
                    "\n",
                    "search_results = index.search(\n",
                    "    question,\n",
                    "    num_results=1\n",
                    ")\n",
                    "\n",
                    "search_results\n"
                ],
                "execution_count": null,
                "outputs": []
            },
            {
                "cell_type": "markdown",
                "metadata": { "id": "#VSC-cbc09088", "language": "markdown" },
                "source": ["Q3. RAG\n"]
            },
            {
                "cell_type": "code",
                "metadata": { "id": "#VSC-ed645b5d", "language": "python" },
                "source": [
                    "from dotenv import load_dotenv\n",
                    "load_dotenv()\n",
                    "\n",
                    "import os\n",
                    "from rag_helper import RAGBase\n",
                    "from google import genai\n",
                    "\n",
                    "client = genai.Client(api_key=os.getenv(\"GEMINI_API_KEY\"))\n",
                    "\n",
                    "assistant = RAGBase(\n",
                    "    index=index,\n",
                    "    llm_client=client\n",
                    ")\n"
                ],
                "execution_count": null,
                "outputs": []
            },
            {
                "cell_type": "code",
                "metadata": { "id": "#VSC-8f35052d", "language": "python" },
                "source": [
                    "answer = assistant.rag(\"How does the agentic loop keep calling the model until it stops?\")\n",
                    "print(answer)\n",
                    "\n",
                    "print(f\"Input tokens: {assistant.last_response.usage_metadata.prompt_token_count}\")\n"
                ],
                "execution_count": null,
                "outputs": []
            },
            {
                "cell_type": "markdown",
                "metadata": { "id": "#VSC-2bc50ecb", "language": "markdown" },
                "source": ["Q4. Chunking\n"]
            },
            {
                "cell_type": "code",
                "metadata": { "id": "#VSC-06321f48", "language": "python" },
                "source": [
                    "from gitsource import chunk_documents\n",
                    "\n",
                    "chunks = chunk_documents(documents, size=2000, step=1000)\n"
                ],
                "execution_count": null,
                "outputs": []
            },
            {
                "cell_type": "code",
                "metadata": { "id": "#VSC-b3a05dac", "language": "python" },
                "source": ["len(chunks)\n"],
                "execution_count": null,
                "outputs": []
            },
            {
                "cell_type": "markdown",
                "metadata": { "id": "#VSC-d2c48f67", "language": "markdown" },
                "source": ["Q5. RAG with chunking\n"]
            },
            {
                "cell_type": "code",
                "metadata": { "id": "#VSC-448bc0ab", "language": "python" },
                "source": [
                    "from minsearch import Index\n",
                    "\n",
                    "index = Index(\n",
                    "    text_fields=[\"content\"],\n",
                    "    keyword_fields=[\"filename\"]\n",
                    ")\n",
                    "\n",
                    "index.fit(chunks)\n",
                    "\n",
                    "from dotenv import load_dotenv\n",
                    "load_dotenv()\n",
                    "\n",
                    "import os\n",
                    "from rag_helper import RAGBase\n",
                    "from google import genai\n",
                    "\n",
                    "client = genai.Client(api_key=os.getenv(\"GEMINI_API_KEY\"))\n",
                    "\n",
                    "assistant = RAGBase(\n",
                    "    index=index,\n",
                    "    llm_client=client\n",
                    ")\n",
                    "\n",
                    "answer = assistant.rag(\"How does the agentic loop keep calling the model until it stops?\")\n",
                    "print(answer)\n",
                    "\n",
                    "print(f\"Input tokens: {assistant.last_response.usage_metadata.prompt_token_count}\")\n"
                ],
                "execution_count": null,
                "outputs": []
            },
            {
                "cell_type": "markdown",
                "metadata": { "id": "#VSC-08a1f2a2", "language": "markdown" },
                "source": ["Q6. Turning it into an agent\n"]
            },
            {
                "cell_type": "code",
                "metadata": { "id": "#VSC-820fa9ac", "language": "python" },
                "source": [
                    "def search(query: str) -> list:\n",
                    "    \"\"\"Search the chunk index database for entries matching the given query.\n",
                    "\n",
                    "    Args:\n",
                    "        query: The search query text to look up in the document chunks.\n",
                    "\n",
                    "    Returns:\n",
                    "        A list of matching document chunks from the minsearch index.\n",
                    "    \"\"\"\n",
                    "    boost_dict = {\"content\": 1.0}\n",
                    "    \n",
                    "    results = index.search(\n",
                    "        query=query,\n",
                    "        num_results=5,\n",
                    "        boost_dict=boost_dict\n",
                    "    )\n",
                    "    \n",
                    "    return results\n"
                ],
                "execution_count": null,
                "outputs": []
            },
            {
                "cell_type": "code",
                "metadata": { "id": "#VSC-fc6b88c1", "language": "python" },
                "source": [
                    "search_tool = {\n",
                    "    \"function_declarations\": [\n",
                    "        {\n",
                    "            \"name\": \"search\",\n",
                    "            \"description\": \"Search the FAQ database for entries matching the given query.\",\n",
                    "            \"parameters\": {\n",
                    "                \"type\": \"OBJECT\",       # Обов'язково великими літерами\n",
                    "                \"properties\": {\n",
                    "                    \"query\": {\n",
                    "                        \"type\": \"STRING\",  # Обов'язково великими літерами\n",
                    "                        \"description\": \"Search query text to look up in the course FAQ.\"\n",
                    "                    }\n",
                    "                },\n",
                    "                \"required\": [\"query\"]\n",
                    "                # Ключ additionalProperties видаляємо, він не підтримується у такому вигляді\n",
                    "            }\n",
                    "        }\n",
                    "    ]\n",
                    "}\n"
                ],
                "execution_count": null,
                "outputs": []
            },
            {
                "cell_type": "code",
                "metadata": { "id": "#VSC-e24a9d30", "language": "python" },
                "source": [
                    "import os\n",
                    "from google import genai\n",
                    "from google.genai import types\n",
                    "\n",
                    "client = genai.Client(api_key=os.getenv(\"GEMINI_API_KEY\"))\n",
                    "\n",
                    "instructions = \"\"\"\n",
                    "You're a course teaching assistant. Answer the student's question using the search tool. \n",
                    "Make multiple searches with different keywords before answering.\n",
                    "\"\"\"\n",
                    "question = \"How does the agentic loop work, and how is it different from plain RAG?\"\n",
                    "\n",
                    "messages = [\n",
                    "    types.Content(role=\"user\", parts=[types.Part.from_text(text=question)])\n",
                    "]\n",
                    "\n",
                    "\n",
                    "search_call_counter = 0\n",
                    "\n",
                    "def search(query: str) -> list:\n",
                    "    \"\"\"Search the chunk index database for entries matching the given query.\"\"\"\n",
                    "    global search_call_counter\n",
                    "    search_call_counter += 1  # Тепер він точно збільшиться!\n",
                    "    \n",
                    "    boost_dict = {\"content\": 1.0}\n",
                    "    return index.search(\n",
                    "        query=query,\n",
                    "        num_results=5,\n",
                    "        boost_dict=boost_dict\n",
                    "    )\n",
                    "\n",
                    "search_call_counter = 0\n",
                    "\n",
                    "# agentic loop\n",
                    "messages = [\n",
                    "    types.Content(role=\"user\", parts=[types.Part.from_text(text=question)])\n",
                    "]\n",
                    "\n",
                    "while True:\n",
                    "    response = client.models.generate_content(\n",
                    "        model=\"gemini-flash-lite-latest\",\n",
                    "        contents=messages,\n",
                    "        config={\n",
                    "            \"system_instruction\": instructions,\n",
                    "            \"tools\": [search_tool]\n",
                    "        }\n",
                    "    )\n",
                    "    \n",
                    "    if response.function_calls:\n",
                    "        call = response.function_calls[0]\n",
                    "        print(f\"[Agent] Викликаю інструмент '{call.name}' з запитом: '{call.args['query']}'\")\n",
                    "        \n",
                    "        # Тут виконається саме та функція, яку ми написали на початку цієї комірки!\n",
                    "        results = search(**call.args)\n",
                    "        \n",
                    "        messages.append(response.candidates[0].content)\n",
                    "        messages.append(\n",
                    "            types.Content(\n",
                    "                role=\"user\",\n",
                    "                parts=[types.Part.from_function_response(name=call.name, response={\"result\": results})]\n",
                    "            )\n",
                    "        )\n",
                    "    else:\n",
                    "        print(\"\\n--- ВІДПОВІДЬ АГЕНТА ---\")\n",
                    "        print(response.text)\n",
                    "        break\n",
                    "print(response.text)\n"
                ],
                "execution_count": null,
                "outputs": []
            },
            {
                "cell_type": "code",
                "metadata": { "id": "#VSC-0e687a2b", "language": "python" },
                "source": ["print(f\"Кількість викликів інструменту search: {search_call_counter}\")\n"],
                "execution_count": null,
                "outputs": []
            }
        ],
        "metadata": {},
        "nbformat": 4,
        "nbformat_minor": 5
    }
                    }
                },
                "required": ["query"]
                # Ключ additionalProperties видаляємо, він не підтримується у такому вигляді
            }
        }
    ]
}

In [20]:
import os
from google import genai
from google.genai import types

client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

instructions = """
You're a course teaching assistant. Answer the student's question using the search tool. 
Make multiple searches with different keywords before answering.
"""
question = "How does the agentic loop work, and how is it different from plain RAG?"

messages = [
    types.Content(role="user", parts=[types.Part.from_text(text=question)])
]


search_call_counter = 0

def search(query: str) -> list:
    """Search the chunk index database for entries matching the given query."""
    global search_call_counter
    search_call_counter += 1  # Тепер він точно збільшиться!
    
    boost_dict = {"content": 1.0}
    return index.search(
        query=query,
        num_results=5,
        boost_dict=boost_dict
    )

search_call_counter = 0

# agentic loop
messages = [
    types.Content(role="user", parts=[types.Part.from_text(text=question)])
]

while True:
    response = client.models.generate_content(
        model="gemini-flash-lite-latest",
        contents=messages,
        config={
            "system_instruction": instructions,
            "tools": [search_tool]
        }
    )
    
    if response.function_calls:
        call = response.function_calls[0]
        print(f"[Agent] Викликаю інструмент '{call.name}' з запитом: '{call.args['query']}'")
        
        # Тут виконається саме та функція, яку ми написали на початку цієї комірки!
        results = search(**call.args)
        
        messages.append(response.candidates[0].content)
        messages.append(
            types.Content(
                role="user",
                parts=[types.Part.from_function_response(name=call.name, response={"result": results})]
            )
        )
    else:
        print("\n--- ВІДПОВІДЬ АГЕНТА ---")
        print(response.text)
        break
print(response.text)

[Agent] Викликаю інструмент 'search' з запитом: 'what is the agentic loop'

--- ВІДПОВІДЬ АГЕНТА ---
The **agentic loop** is a design pattern where an LLM is placed in control of a decision-making process. Unlike a static pipeline, the agent acts as an autonomous driver that can evaluate its own progress, execute actions, and adapt based on the results it receives.

### How the Agentic Loop Works
The agentic loop functions as a continuous cycle that repeats until the model determines it has sufficient information to answer the user's request. It relies on three core components:
1.  **Instructions (System Prompt):** Defines the agent's persona and logic for when to act or when to provide a final answer.
2.  **Tools:** External capabilities (like search functions) that the agent can invoke.
3.  **Memory (Conversation History):** A log of every prompt, tool call, and tool result. The agent uses this history to understand what it has already tried and to avoid repeating errors.

**The cycl

In [21]:
print(f"Кількість викликів інструменту search: {search_call_counter}")

Кількість викликів інструменту search: 1
